# Goal-progress heat maps across tasks, split by brain region

Neurons × normalised goal-progress bins, rank-ordered by peak in Task A, that order
carried to three further panels — now split by the anatomy from `code/histology_refit/`.

Prior art: `LEC_sploratory_analysis.ipynb` cells 84–85 (6 sessions, all mice pooled, no
region split). Two things differ here, both consequential.

**Panel 4 is a within-task repeat, not a fourth task.** In 19 of 25 recdays session 3
re-runs session 0's task. Rather than let that masquerade as a fourth task, it is labelled
and used as a **ceiling**: how much order survives when nothing remapped. Panels 2–3 are
read against it.

**Panel 1 is circular.** The sort is defined on panel 1 and applied to panel 1, so its
diagonal is guaranteed — §1 proves this on pure noise. Read evidence from panels 2–4 only.

In [ ]:
import sys; sys.path.insert(0, '.')
import matplotlib.pyplot as plt
import pandas as pd

import gp_region_heatmaps as gp

pd.set_option('display.width', 220)
print('curves cache :', gp.CURVES_PATH)
print('figures      ->', gp.FIGURE_DIR)

## 1. Synthetic gate — run this before believing any panel

Three populations with known cross-task behaviour, pushed through the *real* plotting and
scoring code:

| planted | expected |
|---|---|
| abstract progress (phase fixed across tasks) | diagonal in all four panels |
| remapping (new phase per task) | panels 1 and 4 only — this is what makes 4 a valid ceiling |
| pure noise | **panel 1 only** |

The noise row is the point: it produces a razor-sharp panel-1 diagonal because panel 1
defines the sort. That is why panel 1 is never evidence.

In [ ]:
gate = gp.run_synthetic_controls()
assert gate.attrs['passed'], 'synthetic gate FAILED — do not interpret the real figure'
gp.plot_synthetic_controls(save=True); plt.show()

## 2. Data gates

Per recday: that panel 4 really is panel 1's task, that panels 2–3 are novel and differ
from each other, and that the positional neuron join to `unit_regions.pkl` holds.

Sessions are chosen by **matching task sequences by value**, never by position — positional
pairing is what created the confound in the first place, and is also what caused the
`ly05` recday bug. Two recdays (`ah08_20250620_20250623`, `ly05_20250618_20250619`) list a
session 3 in `tasks_dic` that has no `Smoothed_norm` behind it, and it is exactly their
repeat session, so they drop out.

In [ ]:
gates = gp.verify()
assert gates.attrs['passed'], 'data gates FAILED'

## 3. Pool the neurons

`Smoothed_norm` is already circularly smoothed (σ=10, wrap-around) and is (n, 360) =
4 states × 90 bins; the four states are averaged to give the 90-bin goal-progress curve.
The curves are cached to `gp_curves.pkl` so this does not re-read the 3.8 GB `data_dic`.

In [ ]:
panels, meta = gp.stack_panels()
tab = pd.crosstab(meta['mouse'], meta['group'])
tab = tab[[c for c in gp.REGION_ORDER if c in tab.columns]]
tab.loc['TOTAL'] = tab.sum()
print(tab.to_string())

## 4. The figure

Rows = brain region (plus an ALL row), columns = the four panels. Per-neuron z-score across
the 90 bins, shared scale ±2, one sort order from Task A applied to all four.

In [ ]:
gp.plot_gp_region_grid(panels, meta, save=True); plt.show()

## 5. The eyeball, as a number

Circular correlation between each neuron's peak bin in Task A and its peak bin in the given
panel, against a within-region shuffle null.

`retention` = mean(novel r) / ceiling r — how much of the ordering that survives a *repeat*
also survives a *task switch*. It is withheld where the ceiling is itself at floor, because
a region that cannot reproduce its own ordering within a task says nothing about remapping.

In [ ]:
summary = gp.summary(panels, meta)
summary.to_csv(gp.FIGURE_DIR / 'diagonality.csv', index=False)
print(summary.to_string(index=False))

## 6. Caveats that belong in any caption

- **Panel 1 carries no evidence** — it defines the sort. §1 shows pure noise producing the
  same diagonal.
- **Panel 4 is Task A again**, so it is a noise ceiling, not a fourth task. The gap between
  panels 2–3 and panel 4 is the remapping effect; the gap between panel 4 and panel 1 is
  session-to-session instability.
- **Rows are not equally powered, and some are one animal.** ENTl-sup is 90% ah08 and ENTm
  67% ly07 — those rows are single-animal results wearing a region label. CA1/HPF has only
  two contributing mice.
- **Units are unit-recordings pooled over recdays, not unique neurons**: a mouse's recdays
  are independent sorts of the same probe in the same brain, so the effective n is far below
  the printed n and the shuffle null does not account for that.
- Region labels inherit ~8–13 µm of registration uncertainty, and ly05's fit is the weakest
  in the cohort (see `code/histology_refit/PROBE_REFIT.md`).